# Grid Search XGBoost para Dataset Adidas

Esta libreta genera múltiples configuraciones `.yaml` en la carpeta compatible con Hydra:
`hfedxgboost/hfedxgboost/conf/xgboost_params_centralized`

Cada configuración se usa para lanzar el script centralizado y evaluar el rendimiento.
Los resultados se mostrarán por pantalla y pueden ser capturados luego en CSV si se desea.

In [ ]:
#!pip install pandas openpyxl
#!pip install matplotlib seaborn
#!pip install qgrid
#!pip install tabulate



In [ ]:
# pip install -r requirements.txt


In [ ]:
import subprocess
from pathlib import Path
import yaml
import itertools
import sys
import pandas as pd

## Preparar datasets

In [ ]:
import subprocess
from pathlib import Path
import sys

prepare_dir = Path("hfedxgboost")
python_path = Path(sys.executable)  # esto apunta a tu Python actual en la libreta

result = subprocess.run(
    [str(python_path), "prepare_adidas.py"],
    cwd=prepare_dir,
    capture_output=True,
    text=True
)

print("📤 STDOUT:\n", result.stdout)
print("\n💥 STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"❌ El script falló con código {result.returncode}")




## Ejecución centralizada

In [ ]:
# Carpeta donde Hydra busca los .yaml (debe existir)
base_dir = Path("hfedxgboost/conf/xgboost_params_centralized")

base_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Grid de hiperparámetros 
param_grid = {
    'n_estimators': [500, 800],
    'max_depth': [8, 10, 12, 15],               # más profundidad
    'subsample': [0.7, 0.9, 1.0],               # valores altos
    'learning_rate': [0.1, 0.05, 0.03],         # bajamos más
    'colsample_bylevel': [1],                   # lo mantenemos fijo
    'colsample_bynode': [1],
    'colsample_bytree': [1],
    'alpha': [0],                               # lo eliminamos (no penalización)
    'gamma': [0, 1],                            # muy bajo (menos penalización)
    'num_parallel_tree': [1],
    'min_child_weight': [1, 3],                 # menor peso mínimo
}


combinaciones = list(itertools.product(*param_grid.values()))
keys = list(param_grid.keys())

for i, values in enumerate(combinaciones):
    config = {k: v for k, v in zip(keys, values)}
    yaml_name = f"adidas_xgb_run_{i:03d}"
    yaml_path = base_dir / f"{yaml_name}.yaml"

    with open(yaml_path, "w") as f:
        yaml.dump(config, f, sort_keys=False)  

    print(f"✔️ Generado: {yaml_path}")

    # Ejecutar el experimento
    print(f"🚀 Ejecutando experimento {i+1}/{len(combinaciones)}...")
    result = subprocess.run([
        str(Path(sys.executable)), "-m", "hfedxgboost.main",
        "--config-name", "Centralized_Baseline",
        "dataset=adidas",
        f"xgboost_params_centralized={yaml_name}"
    ], capture_output=True, text=True)

    print("📤 STDOUT:")
    print(result.stdout)
    print("💥 STDERR:")
    print(result.stderr)



### Estadísticas centralizado

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el CSV de resultados
df = pd.read_csv("results_centralized.csv")

# Ordenar por menor error de test
df_sorted = df.sort_values(by="result_test", ascending=True)

# Mostrar las mejores y peores
print("🏆 Mejores configuraciones:")
display(df_sorted.head(5))

print("💥 Peores configuraciones:")
display(df_sorted.tail(5))

# Plot evolución del error
plt.figure(figsize=(10, 4))
plt.plot(df_sorted["result_test"].values, marker='o')
plt.title("Error de Test por configuración (ordenado)")
plt.ylabel("MSE")
plt.xlabel("Configuración")
plt.grid(True)
plt.show()

# Heatmap de correlación entre hiperparámetros y resultado
param_cols = [
    'xgb_max_depth', 'subsample', 'learning_rate',
    'colsample_bylevel', 'colsample_bynode', 'colsample_bytree',
    'alpha', 'gamma', 'num_parallel_tree', 'min_child_weight'
]

# Crear matriz de correlación
corr_matrix = df[param_cols + ['result_test']].corr()

# Visualizar heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix[["result_test"]].sort_values(by="result_test", ascending=False), 
            annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación de hiperparámetros con el error de test")
plt.show()


## Ejecución federada

In [ ]:
import torch
print(torch.cuda.is_available())  # Debe devolver True
print(torch.cuda.get_device_name(0))  # Nombre de la GPU

In [ ]:
import itertools
import yaml
from pathlib import Path

base_dir = Path("hfedxgboost/conf/clients")
base_dir.mkdir(parents=True, exist_ok=True)

client_num  = 28
num_rounds  = 10

blocks = [
    {"n_estimators_client": 25, "num_iterations": 100},
    {"n_estimators_client": 25, "num_iterations": 500},
    {"n_estimators_client": 50, "num_iterations": 100},
    {"n_estimators_client": 50, "num_iterations": 500},
]

max_depths = [4, 6, 8]
lrs        = [1e-3, 5e-4]

for blk in blocks:
    for depth, lr in itertools.product(max_depths, lrs):
        cfg = {
            "n_estimators_client": blk["n_estimators_client"],
            "num_rounds":         num_rounds,
            "client_num":         client_num,
            "num_iterations":     blk["num_iterations"],
            "xgb":   {"max_depth": depth},
            "CNN":   {"lr": lr}
        }

        # Damos un nombre único que incluya todos los parámetros
        lr_str = f"{lr:.4f}".rstrip("0").rstrip(".")
        yaml_name = (
            f"adidas_{client_num}"
            f"_n{blk['n_estimators_client']}"
            f"_it{blk['num_iterations']}"
            f"_xgb{depth}"
            f"_cnn{lr_str}.yaml"
        )
        yaml_path = base_dir / yaml_name

        with open(yaml_path, "w") as f:
            yaml.dump(cfg, f, sort_keys=False)

        print(f"✔️ Generado: {yaml_path}")


Ahora simplemente, por facilidad podríamos lanzar todos los experimentos, en el terminal con el comando de multirun:

python -m hfedxgboost.main --multirun \
  clients=adidas_28_n25_it100_xgb4_cnn0.001,adidas_28_n25_it100_xgb4_cnn0.0005,adidas_28_n25_it100_xgb6_cnn0.001,adidas_28_n25_it100_xgb6_cnn0.0005,adidas_28_n25_it100_xgb8_cnn0.001,adidas_28_n25_it100_xgb8_cnn0.0005,adidas_28_n25_it500_xgb4_cnn0.001,adidas_28_n25_it500_xgb4_cnn0.0005,adidas_28_n25_it500_xgb6_cnn0.001,adidas_28_n25_it500_xgb6_cnn0.0005,adidas_28_n25_it500_xgb8_cnn0.001,adidas_28_n25_it500_xgb8_cnn0.0005,adidas_28_n50_it100_xgb4_cnn0.001,adidas_28_n50_it100_xgb4_cnn0.0005,adidas_28_n50_it100_xgb6_cnn0.001,adidas_28_n50_it100_xgb6_cnn0.0005,adidas_28_n50_it100_xgb8_cnn0.001,adidas_28_n50_it100_xgb8_cnn0.0005,adidas_28_n50_it500_xgb4_cnn0.001,adidas_28_n50_it500_xgb4_cnn0.0005,adidas_28_n50_it500_xgb6_cnn0.001,adidas_28_n50_it500_xgb6_cnn0.0005,adidas_28_n50_it500_xgb8_cnn0.001,adidas_28_n50_it500_xgb8_cnn0.0005 \
  dataset=adidas


### Análisis de estadísticas

In [ ]:
import pandas as pd

# 1. Carga
df = pd.read_csv('results.csv')

# 2. 5 mejores (menor best_res)
mejores = df.nsmallest(5, 'best_res')

# 3. 5 peores (mayor best_res)
peores = df.nlargest(5, 'best_res')

# 4. Mostrar sólo columnas de interés
cols = ['n_estimators_client','num_iterations','xgb_max_depth','cnn_lr','best_res','best_res_round_num']

print("🏆 5 MEJORES CONFIGURACIONES (menor best_res):")
print(mejores[cols].to_string(index=False))

print("\n💥 5 PEORES CONFIGURACIONES (mayor best_res):")
print(peores[cols].to_string(index=False))





In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('results.csv')

# Marca si el mejor resultado ocurrió exactamente en la última ronda de boosting (num_rounds=10)
df['hit_boost_max'] = df['best_res_round_num'] == df['num_rounds']

# Proporción global
prop_boost_max = df['hit_boost_max'].mean()
print(f"Proporción de experimentos que llegaron al tope de boosting rounds (10): {prop_boost_max:.1%}")

# Por bloque de num_iterations
group_iter = (
    df
    .groupby('num_iterations')
    .agg(
        count=('best_res','size'),
        mean_res=('best_res','mean'),
        prop_boost_max=('hit_boost_max','mean')
    )
    .reset_index()
)
print("\nEstadísticos por num_iterations:\n", group_iter.to_string(index=False))

# T-test para ver si la proporción de tope difiere entre 100 y 500 federated iterations
p_100 = df[df['num_iterations']==100]['hit_boost_max']
p_500 = df[df['num_iterations']==500]['hit_boost_max']
t_stat, p_val = stats.ttest_ind(p_100, p_500, equal_var=False)
print(f"\nT-test hit_boost_max 100 vs 500 iters: t={t_stat:.2f}, p={p_val:.3f}")




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

# 1) Carga de datos
df = pd.read_csv('results.csv')  # ajusta ruta si hace falta

# 2) Matriz de correlaciones + heatmap
corr = df[['n_estimators_client','num_iterations','xgb_max_depth','cnn_lr','best_res']].corr()
fig, ax = plt.subplots()
im = ax.imshow(corr.values, aspect='equal')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.index)
plt.colorbar(im, ax=ax, label='corr')
ax.set_title("Heatmap de correlaciones")
plt.tight_layout()
plt.show()

# 3) Scatter matrix para ver interacciones parejas
fig = plt.figure(figsize=(8,8))
scatter_matrix(
    df[['n_estimators_client','xgb_max_depth','cnn_lr','best_res']],
    alpha=0.6, diagonal='kde',
    ax=plt.gca()
)
plt.suptitle("Scatter matrix de parámetros vs best_res")
plt.tight_layout()
plt.show()

# 4) Heatmaps de best_res medio para combinaciones depth × lr,
#    separados por num_iterations
for it in sorted(df['num_iterations'].unique()):
    pivot = df[df['num_iterations']==it].pivot_table(
        index='xgb_max_depth',
        columns='cnn_lr',
        values='best_res',
        aggfunc='mean'
    )
    fig, ax = plt.subplots()
    im = ax.imshow(pivot.values, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('cnn_lr')
    ax.set_ylabel('xgb_max_depth')
    ax.set_title(f"Mean best_res (depth vs lr) — {it} iter")
    plt.colorbar(im, ax=ax, label='best_res')
    plt.show()

# 5) Boxplots de best_res según n_estimators_client
fig, ax = plt.subplots()
groups = [grp['best_res'].values for _, grp in df.groupby('n_estimators_client')]
labels = [int(g) for g in sorted(df['n_estimators_client'].unique())]
ax.boxplot(groups, labels=labels)
ax.set_xlabel('n_estimators_client')
ax.set_ylabel('best_res')
ax.set_title('Distribución de best_res por n_estimators_client')
plt.show()



## 50 Rondas de mejor configuración

In [ ]:
import subprocess
import sys

# Nombre de tu config de clientes (sin extensión .yaml)
clients_cfg = "adidas_28_clients"

# Construimos el comando Hydra
cmd = [
    sys.executable, "-m", "hfedxgboost.main",
    f"clients={clients_cfg}",         # Carga conf/clients/adidas_28_clients.yaml
    "dataset=adidas",                 # Dataset Adidas
]

print("🚀 Ejecutando experimento federado con 50 rondas…")
result = subprocess.run(cmd, capture_output=True, text=True)

print("📤 STDOUT:\n", result.stdout)
print("💥 STDERR:\n", result.stderr)
